In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision # library for cv
from torchvision.datasets import CIFAR10

## Preprocessing the image

In [4]:
# dataset & dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([  # image => scale(0,1) => normalize (-1,1)
    transforms.ToTensor(), # it will transform the images into tensors and also do scaling 
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # ((means),(std deviations))
    
])

# datasets

trainset = CIFAR10(root="./data_folder",train=True,download=True,transform=transform)
testset = CIFAR10(root="./data_folder",train=False,download=True,transform=transform)

100%|█████████████████████████████████████████| 170M/170M [14:02<00:00, 202kB/s]


In [5]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data_folder
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [6]:
# dataloader

trainloader = DataLoader(trainset,batch_size=64,shuffle=True)
testloader = DataLoader(testset,batch_size=64)

## Build the CNN

In [17]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_layers = nn.Sequential(
            
            # 1st layer
            nn.Conv2d(3,32,kernel_size=3,padding=1), # convo2D - as input would be a matrix , convo2D(input chanels,output chanels,_,_)
            nn.ReLU(),
            nn.MaxPool2d(2,2), # (kernel_size=2,stride=2)

            # 2nd layer
            nn.Conv2d(32,64,kernel_size=3,padding=1),  # we double the number of filters with each layer
            nn.ReLU(),
            nn.MaxPool2d(2,2), 

            # 3rd layer
            nn.Conv2d(64,128,kernel_size=3,padding=1),  
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )

        self.fc_layers = nn.Sequential(

            # hidden layer
            nn.Linear(4*4*128,256), # (input , output) , we randomly choose no. of output 
            nn.ReLU(),

            # output layer 
            nn.Linear(256,10) # 10 classes 
        )

    def forward(self,x):
        x = self.conv_layers(x) # we have the output of convo layers stored in x , we have to flatten it 
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)

        return x

In [18]:
model = CNN()

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Train the model

In [21]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images , labels in trainloader:
        optimizer.zero_grad()

        outputs = model.forward(images) # forwardpropogation
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch} & loss = {epoch_training_loss/len(trainloader)}")
        

epoch=0 & loss = 0.5175181677007614
epoch=1 & loss = 0.42394735212521173
epoch=2 & loss = 0.33399276521123583
epoch=3 & loss = 0.2628499116710461
epoch=4 & loss = 0.20395427781259617
epoch=5 & loss = 0.15675030847120544
epoch=6 & loss = 0.12666325565770536
epoch=7 & loss = 0.11233486027499694
epoch=8 & loss = 0.10153798418967507
epoch=9 & loss = 0.0862155772609841


In [22]:
# evaluate 

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images , labels in testloader:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels +=  labels.size(0)

print(f"accuracy = {correct_labels/total_labels * 100}")

accuracy = 75.74
